In [1]:
import torch
import sys
from pathlib import Path
import torch, torch.nn as nn
import torchvision.models as models

# Add the project root (parent of NoteBooks) to Python's path
project_root = Path.cwd().parent  # if notebook is in NoteBooks/
sys.path.insert(0, str(project_root))

from src.training.train_cnn import create_cnn_model
from src.models.mlp import create_mlp_model_from_splits
from src.data.loader import *
from src.data.preprocessing import * # need thsi for just a sec. 
from scripts.ensemble_evalute import final_evaluate_mlp_on_test, final_evaluate_cnn_on_test
import numpy as np
import random 
from src.utils.model_registry import ModelRegistry # just get it all for now. 
from src.data.preprocessing import *

In [2]:
# constants. 
projections_path = "../IU-xray/indiana_projections.csv"
reports_path = "../IU-xray\indiana_reports.csv"
image_dir = '../IU-xray/images/images_normalized'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = Path("../models")
# where we can find the fetchers later. 
SPLITS_DIR = Path("../data/splits")
MODELS_DIR = Path("../models")

In [ ]:
# create the data splits for training and validation purposes. Trust. 
modes = ["abnormal", "pneumonia"]
for mode in modes:
    create_train_test_split(projections_path, reports_path, mode)

  Train:      2672 (63.7% pos)
  Validation:  573 (63.7% pos)
  Test:        573 (63.7% pos)

✅ All splits saved to ..\data\splits
  Train:      2672 (7.6% pos)
  Validation:  573 (7.5% pos)
  Test:        573 (7.5% pos)

✅ All splits saved to ..\data\splits


In [4]:
def set_seed(seed=42):
    """Set all random seeds for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    # For deterministic behavior (slower but reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

registry = ModelRegistry("../models/registry.json")


In [5]:
# modes = ["abnormal", "pneumonia"]
# seeds = [42, 123, 456, 789, 1024, 2048, 3407, 5555, 7777, 9999]
modes = ["pneumonia"]
seeds = [42]


for mode in modes:
    for seed in seeds:
        # don't do this new everytime, just read the one that is already there. 
        # df = build_master_df_rich(projections_path, reports_path, mode) # get EVERYTHING. 

        # this should do the trick
        train_df = pd.read_csv(SPLITS_DIR / f"{mode}_train.csv")
        val_df = pd.read_csv(SPLITS_DIR / f"{mode}_val.csv")
        test_df = pd.read_csv(SPLITS_DIR / f"{mode}_test.csv")  # Keep untouched

        # subset = True
        # if subset == True:
        #     train_df = train_df.sample(n=200, random_state=42)
        #     val_df = val_df.sample(n=50, random_state=42)
        #     test_df = test_df.sample(n=50, random_state=42)

        # 1. Train MLP
        print("Training MLP's")
        set_seed(seed)
        mlp_model_name = f"mlp_{mode}"
        text_amount = "medium" # use the medium amount of text data
        model_type = "single" # not ensemble, single mode. 
        # returns a state dict, not the model
        MLP_model_state_dict, errors_train, errors_test, losses_train, losses_test = create_mlp_model_from_splits(train_df, val_df, test_df, mode, text_amount, model_type)
        mlp_test_metrics = final_evaluate_mlp_on_test(MLP_model_state_dict, mode, text_amount)
        mlp_auc = mlp_test_metrics["auc"]
        mlp_should_save, mlp_reason, mlp_improvement = registry.should_save(mlp_model_name, mlp_auc)

        # 2. Train CNN
        # print("training finetuned CNN's sire ")
        # cnn_model_name = f"cnn_{mode}"
        # current_best = registry.get_best_auc(cnn_model_name)
        # print(f"Current best AUC for {cnn_model_name}: {current_best:.4f}")
        # set_seed(seed)
        # num_epochs = 25 # YIUPEEE
        # CNN_model_state_dict, history = create_cnn_model(train_df, val_df, image_dir, device=device, mode=mode, num_epochs=num_epochs)
        # cnn_test_metrics = final_evaluate_cnn_on_test(CNN_model_state_dict, mode, image_dir, device)
        # cnn_auc = cnn_test_metrics["auc"]

        # cnn_should_save, cnn_reason, cnn_improvement = registry.should_save(cnn_model_name, cnn_auc)


        if mlp_should_save:
            MLP_filename = f"best_mlp_{mode}_{text_amount}.pth"
            MLP_full_path = save_path / MLP_filename
            torch.save(MLP_model_state_dict, MLP_full_path)

            registry.register_model(
                mlp_model_name, 
                mlp_auc,
                MLP_full_path,
            )


        # if cnn_should_save:
        #     CNN_filename = f"best_cnn_densenet_{mode}.pth"
        #     CNN_full_path = save_path / CNN_filename
        #     torch.save(CNN_model_state_dict, CNN_full_path)

        #     registry.register_model(
        #         cnn_model_name, 
        #         cnn_auc, 
        #         CNN_full_path,
        #         None
        #     )
    
# registry.list_models()


Training MLP's
  Val:   (573, 1000)
  Test:  (573, 1000)
Vectorizer saved to: ../models/vectorizer_pneumonia_medium.pkl
Training MLP

FINAL MLP TEST EVALUATION - PNEUMONIA
Test set: 573 samples (7.5% pos)
✅ Loaded vectorizer from: ..\models\vectorizer_pneumonia_medium.pkl
Features: (573, 1000)
Model input dimension: 1000

************************************
  t=0.50 | Sensitivity: 65.1% | Specificity: 96.4% | FN: 15 | FP: 19
  t=0.30 | Sensitivity: 69.8% | Specificity: 94.2% | FN: 13 | FP: 31
  t=0.20 | Sensitivity: 69.8% | Specificity: 93.4% | FN: 13 | FP: 35
  t=0.10 | Sensitivity: 79.1% | Specificity: 91.3% | FN: 9 | FP: 46
  t=0.05 | Sensitivity: 81.4% | Specificity: 89.8% | FN: 8 | FP: 54

Test Accuracy : 94.1%
Test AUC-ROC  : 0.9317

Confusion Matrix:
  True Negatives  (correctly said normal)   : 511
  False Positives (incorrectly flagged)      : 19
  False Negatives (missed abnormal cases)   : 15  ← this is the dangerous one
  True Positives  (correctly caught)         : 28

Se

In [6]:
import torchxrayvision as xrv
import numpy as np

def load_chexpert_model(device="cuda"):
    print("loading chexpert...")

    model = xrv.models.DenseNet(weihtts="densenet121-res224-all")

    model = model.to(device)
    model.eval() 

    return model

In [7]:
"""
test_chexpert_baseline.py
Load a CheXpert-pretrained model and evaluate on your dataset.
"""
import torch
import torch.nn as nn
import torchxrayvision as xrv
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score

def load_chexpert_model(device="cuda"):
    """
    Load CheXpert-pretrained DenseNet-121.
    Trained on 224,316 chest X-rays from CheXpert dataset.
    """
    print("📦 Loading CheXpert-pretrained model...")
    
    # Load the model (trained on CheXpert + NIH + PC)
    model = xrv.models.DenseNet(weights="densenet121-res224-all")
    
    # The model expects single-channel input originally
    # But we can adapt it to work with our 1-channel images
    model = model.to(device)
    model.eval()
    
    print("✅ CheXpert model loaded!")
    print(f"   Trained on: CheXpert (224K), NIH (112K), PC (178K)")
    print(f"   Classes: {model.pathologies}")  # 14 pathologies
    
    return model

def evaluate_chexpert_on_your_data(model, test_df, image_dir, device="cuda"):
    """
    Evaluate CheXpert model on your test data.
    Note: CheXpert has 14 outputs, we need to extract the pneumonia one.
    """
    from src.data.dataset import ChestXRayDataset
    from src.data.loader import get_transforms_grayscale
    from torch.utils.data import DataLoader
    
    _, test_transforms = get_transforms_grayscale()
    test_dataset = ChestXRayDataset(test_df, image_dir, transform=test_transforms)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    all_probs = []
    all_labels = []
    
    # Find which index is "Pneumonia" in CheXpert's output
    pneumonia_idx = model.pathologies.index("Pneumonia") if "Pneumonia" in model.pathologies else None
    print(f"Pneumonia index in CheXpert: {pneumonia_idx}")
    print(f"All pathologies: {model.pathologies}")
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            
            # CheXpert model expects 1-channel input
            # Our images are already 1-channel from transforms
            
            outputs = model(images)  # Shape: (batch, 14) - 14 pathologies
            
            # Extract pneumonia probability
            pneumonia_logits = outputs[:, pneumonia_idx]
            pneumonia_probs = torch.sigmoid(pneumonia_logits).cpu().numpy()
            
            all_probs.extend(pneumonia_probs)
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate AUC
    auc = roc_auc_score(all_labels, all_probs)
    
    print(f"\n{'='*60}")
    print(f"CHEXPERT BASELINE ON YOUR DATA")
    print(f"{'='*60}")
    print(f"Pneumonia AUC: {auc:.4f}")
    print(f"Your best CNN: 0.6392")
    print(f"Difference: {auc - 0.6392:+.4f}")
    
    if auc > 0.70:
        print("\n✅ CheXpert outperforms your model → Your training can improve!")
    else:
        print("\n⚠️ CheXpert also struggles → Data/label quality is the bottleneck")
    
    return auc, all_probs, all_labels


def test_chexpert_multiple_thresholds(all_probs, all_labels):
    """Test CheXpert at different thresholds."""
    from sklearn.metrics import confusion_matrix
    
    print("\n📊 CheXpert Threshold Analysis:")
    for t in [0.5, 0.3, 0.2, 0.1, 0.05]:
        preds = (np.array(all_probs) >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        print(f"  t={t:.2f} | Sens: {sens:.1%} | Spec: {spec:.1%} | FN: {fn:3d} | FP: {fp:3d}")


def evaluate_chexpert_correctly(test_df, image_dir, device="cuda"):
    """
    Evaluate CheXpert with PROPER normalization.
    """
    import torchxrayvision as xrv
    import numpy as np
    from PIL import Image
    from sklearn.metrics import roc_auc_score
    
    # Load model
    model = xrv.models.DenseNet(weights="densenet121-res224-all")
    model = model.to(device)
    model.eval()
    
    pneumonia_idx = model.pathologies.index("Pneumonia")
    
    all_probs = []
    all_labels = []
    
    for idx, row in test_df.iterrows():
        # Load image
        img_path = Path(image_dir) / row['filename']
        
        # ✅ Load as grayscale PIL Image
        img = Image.open(img_path).convert('L')
        
        # ✅ Convert to numpy array in [0, 255]
        img_np = np.array(img, dtype=np.float32)
        
        # ✅ Denormalize: [-1, 1] → [0, 255]
        # Your images are normalized with mean=0.5, std=0.5
        # Which maps [0, 1] → [-1, 1]
        # We need to reverse this
        img_np = (img_np * 0.5 + 0.5) * 255  # Now in [0, 255]
        
        # ✅ Convert to CheXpert's expected range [-1024, 1024]
        img_np = img_np * 8.03 - 1024  # Scale to approximate HU units
        # OR simply: img_np = img_np - 1024  (shift to match)
        
        # Resize to 224x224
        from skimage.transform import resize
        img_resized = resize(img_np, (224, 224), preserve_range=True)
        
        # Add batch and channel dimensions
        img_tensor = torch.tensor(img_resized, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        img_tensor = img_tensor.to(device)
        
        # Predict
        with torch.no_grad():
            outputs = model(img_tensor)
            pneumonia_prob = torch.sigmoid(outputs[0, pneumonia_idx]).cpu().item()
        
        all_probs.append(pneumonia_prob)
        all_labels.append(row['label'])
        
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx+1}/{len(test_df)}")
    
    auc = roc_auc_score(all_labels, all_probs)
    
    print(f"\n{'='*60}")
    print(f"CHEXPERT WITH CORRECT NORMALIZATION")
    print(f"{'='*60}")
    print(f"Pneumonia AUC: {auc:.4f}")
    
    # Check prediction distribution
    probs_array = np.array(all_probs)
    print(f"Prediction mean: {probs_array.mean():.4f}")
    print(f"Prediction std:  {probs_array.std():.4f}")
    print(f"Prediction min:  {probs_array.min():.4f}")
    print(f"Prediction max:  {probs_array.max():.4f}")
    
    return auc, all_probs, all_labels


# Run the test
# if __name__ == "__main__":
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
#     # Load your test data
#     test_df = pd.read_csv("../data/splits/pneumonia_test.csv")
    
#     # Load CheXpert model
#     chexpert_model = load_chexpert_model(device)
    
#     # Evaluate
#     # auc, probs, labels = evaluate_chexpert_on_your_data(
#     #     chexpert_model, test_df, image_dir, device
#     # )
    
#     # test_chexpert_multiple_thresholds(probs, labels)

#     evaluate_chexpert_correctly(test_df, image_dir)